In [3]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("⚠️ 런타임 유형을 GPU(T4 이상)로 변경해 주세요!")

Mounted at /content/drive
PyTorch Version: 2.14.1+cu130
CUDA Available: True
GPU Device: Tesla T4


In [4]:
!pip install -q transformers safetensors soundfile tqdm pandas scikit-learn

In [6]:
import os
import zipfile

# 본인 구글 드라이브 내 zip 경로 (필요시 수정)
zip_path = '/content/drive/MyDrive/대학부 데이터-20260830T163301Z-1-001.zip'
if not os.path.exists(zip_path):
    # 드라이브 내 탐색
    for root, dirs, files in os.walk('/content/drive/MyDrive'):
        for f in files:
            if '대학부 데이터' in f and f.endswith('.zip'):
                zip_path = os.path.join(root, f)
                break
        if os.path.exists(zip_path):
            break

print("대상 Zip 파일:", zip_path)

os.makedirs('/content/data', exist_ok=True)
print("Colab 고속 디스크로 압축 해제 중...")
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/data')
print("압축 해제 완료!")

대상 Zip 파일: /content/drive/MyDrive/대학부 데이터-20260830T163301Z-1-001.zip
Colab 고속 디스크로 압축 해제 중...
압축 해제 완료!


In [7]:
import os
import json
import pandas as pd
from tqdm import tqdm

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]

# 라벨 폴더 경로 자동 탐색
label_dir = None
for root, dirs, files in os.walk('/content/data'):
    if 'TL_서울_구급' in root or ('라벨링데이터' in root and any(f.endswith('.json') for f in files)):
        label_dir = root
        break

if not label_dir:
    raise FileNotFoundError("라벨 json 폴더를 찾을 수 없습니다. 경로를 확인하세요.")
print("라벨 디렉토리:", label_dir)

json_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.json')])
print(f"발견된 라벨 JSON: {len(json_files)}건")

augmented_rows = []
for fn in tqdm(json_files, desc="Prefix 증강 데이터 생성"):
    fp = os.path.join(label_dir, fn)
    with open(fp, 'r', encoding='utf-8') as f:
        d = json.load(f)

    raw = d.get('symptom') or []
    valid = [s for s in TARGET_SYMPTOMS if s in raw]
    if not valid:
        continue  # 빈 증상 샘플은 대회 규정상 제외

    utts = d.get('utterances') or []
    utts.sort(key=lambda u: int(u.get('startAt', 0)))
    if not utts:
        continue

    # 단계별 발화 슬라이싱 지점 (3턴, 5턴, 8턴, 12턴, 통화 전체)
    cutoffs = [3, 5, 8, 12, len(utts)]
    seen_cutoffs = set()

    for c in cutoffs:
        c = min(c, len(utts))
        if c in seen_cutoffs:
            continue
        seen_cutoffs.add(c)

        # 1~c번째 발화 텍스트 결합
        prefix_text = " ".join(u.get('text', '').strip() for u in utts[:c] if u.get('text'))
        if not prefix_text.strip():
            continue

        row = {
            'call_id': fn,
            'cutoff_step': c,
            'text': prefix_text
        }
        for s in TARGET_SYMPTOMS:
            row[s] = 1 if s in valid else 0
        augmented_rows.append(row)

df = pd.DataFrame(augmented_rows)
print(f"\n[데이터셋 준비 완료]")
print(f"- 원본 통화 수: {len(json_files)}건")
print(f"- 증강된 학습 샘플 수: {len(df)}건")
df.to_csv('/content/augmented_train.csv', index=False, encoding='utf-8-sig')

라벨 디렉토리: /content/data/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급
발견된 라벨 JSON: 2055건


Prefix 증강 데이터 생성: 100%|██████████| 2055/2055 [00:00<00:00, 4016.59it/s]


[데이터셋 준비 완료]
- 원본 통화 수: 2055건
- 증강된 학습 샘플 수: 10246건


In [9]:
import os
import json
import pandas as pd
from tqdm import tqdm

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]

# 라벨 폴더 경로 자동 탐색
label_dir = None
for root, dirs, files in os.walk('/content/data'):
    if 'TL_서울_구급' in root or ('라벨링데이터' in root and any(f.endswith('.json') for f in files)):
        label_dir = root
        break

if not label_dir:
    raise FileNotFoundError("라벨 json 폴더를 찾을 수 없습니다.")
print("라벨 디렉토리:", label_dir)

json_files = sorted([f for f in os.listdir(label_dir) if f.endswith('.json')])

augmented_rows = []
for fn in tqdm(json_files, desc="화자 태그 포함 Prefix 증강 데이터 생성"):
    fp = os.path.join(label_dir, fn)
    with open(fp, 'r', encoding='utf-8') as f:
        d = json.load(f)

    raw = d.get('symptom') or []
    valid = [s for s in TARGET_SYMPTOMS if s in raw]
    if not valid:
        continue  # 빈 증상 샘플은 대회 규정상 제외

    utts = d.get('utterances') or []
    utts.sort(key=lambda u: int(u.get('startAt', 0)))
    if not utts:
        continue

    # 1. 턴별 화자 태그 결합 함수 (대원=0, 신고자=1)
    def format_utt(u):
        spk = int(u.get('speaker', -1))
        tag = "[대원] " if spk == 0 else ("[신고자] " if spk == 1 else "")
        txt = u.get('text', '').strip()
        return f"{tag}{txt}" if txt else ""

    # 2. 단계별 발화 슬라이싱 지점 (3턴, 5턴, 8턴, 12턴, 통화 전체)
    cutoffs = [3, 5, 8, 12, len(utts)]
    seen_cutoffs = set()

    for c in cutoffs:
        c = min(c, len(utts))
        if c in seen_cutoffs:
            continue
        seen_cutoffs.add(c)

        prefix_texts = [format_utt(u) for u in utts[:c]]
        prefix_text = " ".join(t for t in prefix_texts if t).strip()
        if not prefix_text:
            continue

        row = {
            'call_id': fn,
            'cutoff_step': c,
            'text': prefix_text
        }
        for s in TARGET_SYMPTOMS:
            row[s] = 1 if s in valid else 0
        augmented_rows.append(row)

df = pd.DataFrame(augmented_rows)
print(f"\n[데이터셋 준비 완료]")
print(f"- 원본 통화 수: {len(json_files)}건")
print(f"- 화자 태그 증강 샘플 수: {len(df)}건")
print("샘플 예시:\n", df['text'].iloc[1][:120], "...")
df.to_csv('/content/augmented_train_speaker.csv', index=False, encoding='utf-8-sig')

라벨 디렉토리: /content/data/대학부 데이터/Validation/2.라벨링데이터/VL_서울_구급


화자 태그 포함 Prefix 증강 데이터 생성: 100%|██████████| 2055/2055 [00:01<00:00, 1382.14it/s]



[데이터셋 준비 완료]
- 원본 통화 수: 2055건
- 화자 태그 증강 샘플 수: 10246건
샘플 예시:
 [대원] 네, 119입니다. 여보세요. [신고자] 그 장안동 [신고자] ***[개인정보] [대원] 장안동. [신고자] ***[개인정보] ...


In [10]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

TARGET_SYMPTOMS = ["고열", "구토", "두통", "복통", "어지러움", "열상", "오심", "전신쇠약", "호흡곤란"]
MODEL_NAME = "klue/roberta-base"

BATCH_SIZE = 32      # 속도 극대화
EPOCHS = 1           # 1차에서 검증된 최적 에폭
LR = 3e-5
MAX_LEN = 512
POS_WEIGHT_POWER = 0.35  # 0.5 -> 0.35로 조정하여 과잉 오탐(FP) 억제

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = "left"

df = pd.read_csv('/content/augmented_train_speaker.csv')

# 통화 단위 분할 (데이터 누수 차단)
unique_calls = df['call_id'].unique()
train_calls, val_calls = train_test_split(unique_calls, test_size=0.15, random_state=42)

train_df = df[df['call_id'].isin(train_calls)].reset_index(drop=True)
val_df = df[df['call_id'].isin(val_calls)].reset_index(drop=True)
print(f"Train 통화: {len(train_calls)}건 ({len(train_df)}샘플) / Val 통화: {len(val_calls)}건 ({len(val_df)}샘플)")

class SymptomDataset(Dataset):
    def __init__(self, data, tokenizer, max_len):
        self.texts = data['text'].tolist()
        self.labels = data[TARGET_SYMPTOMS].values.astype(np.float32)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        inputs = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_loader = DataLoader(SymptomDataset(train_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SymptomDataset(val_df, tokenizer, MAX_LEN), batch_size=BATCH_SIZE)

# pos_weight = (음성/양성)^0.35
num_pos = train_df[TARGET_SYMPTOMS].sum().values
num_neg = len(train_df) - num_pos
pos_weight = (num_neg / np.maximum(num_pos, 1)) ** POS_WEIGHT_POWER
pos_weight_tensor = torch.tensor(pos_weight, dtype=torch.float32).to(device)

print("조정된 증상별 pos_weight (p=0.35):")
for s, w in zip(TARGET_SYMPTOMS, pos_weight):
    print(f"  - {s}: {w:.2f}")

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(TARGET_SYMPTOMS))
model.to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_tensor)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)

# 1. 학습 진행 (약 12~14분)
model.train()
total_train_loss = 0
for batch in tqdm(train_loader, desc="Training (Speaker + p0.35)"):
    optimizer.zero_grad()
    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    labels = batch['labels'].to(device)

    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    loss = criterion(outputs.logits, labels)
    loss.backward()
    optimizer.step()
    scheduler.step()
    total_train_loss += loss.item()

print(f"\nTrain Loss: {total_train_loss / len(train_loader):.4f}")

# 2. 모델 로컬 저장
save_dir = '/content/best_speaker_p35_model'
os.makedirs(save_dir, exist_ok=True)
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print("Colab 로컬 저장 완료!")

# 3. 구글 드라이브 즉시 자동 백업
drive_model_path = '/content/drive/MyDrive/DCC_best_speaker_p35_model'
os.system(f"cp -r {save_dir} {drive_model_path}")
print(f"✅ 구글 드라이브 백업 완료: {drive_model_path}")

# 4. 4-2용 최신 확률표 추출 및 드라이브 저장
model.eval()
dump_rows = []
with torch.no_grad():
    for _, row in tqdm(val_df.iterrows(), total=len(val_df), desc="4-2용 최신 확률 테이블 추출"):
        inputs = tokenizer(
            str(row['text']),
            max_length=MAX_LEN,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        logits = model(**inputs).logits
        probs = torch.sigmoid(logits).cpu().numpy()[0]

        record = {'call_id': row['call_id'], 'step': row['cutoff_step']}
        for i, s in enumerate(TARGET_SYMPTOMS):
            record[s] = float(probs[i])
        dump_rows.append(record)

prob_df = pd.DataFrame(dump_rows)
prob_df.to_csv('/content/val_probabilities_v2.csv', index=False, encoding='utf-8-sig')
os.system("cp /content/val_probabilities_v2.csv /content/drive/MyDrive/DCC_val_probabilities_v2.csv")
print("🎉 2차 실험 완료! 드라이브에 'DCC_best_speaker_p35_model'과 'DCC_val_probabilities_v2.csv'가 저장되었습니다.")

Using device: cuda
Train 통화: 1746건 (8706샘플) / Val 통화: 309건 (1540샘플)
조정된 증상별 pos_weight (p=0.35):
  - 고열: 1.65
  - 구토: 1.91
  - 두통: 2.09
  - 복통: 1.54
  - 어지러움: 1.58
  - 열상: 1.95
  - 오심: 2.01
  - 전신쇠약: 1.69
  - 호흡곤란: 1.91


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: klue/roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Training (Speaker + p0.35): 100%|██████████| 273/273 [13:35<00:00,  2.99s/it]


Train Loss: 0.6096


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Colab 로컬 저장 완료!
✅ 구글 드라이브 백업 완료: /content/drive/MyDrive/DCC_best_speaker_p35_model


4-2용 최신 확률 테이블 추출: 100%|██████████| 1540/1540 [00:21<00:00, 71.67it/s]

🎉 2차 실험 완료! 드라이브에 'DCC_best_speaker_p35_model'과 'DCC_val_probabilities_v2.csv'가 저장되었습니다.
